# 🛍️ Project 03: Customer Lifetime Value (CLV) & RFM Cohort Segmentation
### Advanced Customer Analytics, Unsupervised Clustering & Revenue Optimization

**Author:** Data Science Portfolio Team  
**Difficulty:** 🟢 Beginner  
**Domain:** E-Commerce & Growth Analytics  

---
### Notebook Outline:
1. **Environment Setup**
2. **Ingestion of Transaction Records**
3. **RFM Metrics Formulation & Distribution Analysis**
4. **Logarithmic & Power Transformations (Mitigating Right Skew)**
5. **K-Means Clustering with Elbow Method & Silhouette Analysis**
6. **Agglomerative Hierarchical Clustering Comparison**
7. **Cluster Persona Profiling & Business Interpretation**
8. **Customer Lifetime Value (CLV) Forecasting & Retention Matrix**

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("E-commerce analytics initialized.")

In [ ]:
# Ingestion & Distribution Inspection
df = pd.read_csv("data/ecommerce_transactions.csv")
print(f"Customer Profiles: {len(df)}")
display(df.describe().T[['mean', 'std', 'min', '50%', 'max']])

In [ ]:
# Visualizing Raw vs Log-Transformed RFM Distributions
fig, axes = plt.subplots(2, 3, figsize=(16, 8))

for idx, col in enumerate(['recency_days', 'frequency_orders', 'monetary_value_usd']):
    sns.histplot(df[col], kde=True, ax=axes[0, idx], color='navy')
    axes[0, idx].set_title(f"Raw {col}", fontweight='bold')
    
    # Log transform to alleviate severe right skew
    sns.histplot(np.log1p(df[col]), kde=True, ax=axes[1, idx], color='teal')
    axes[1, idx].set_title(f"Log1p({col})", fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Preprocessing: Log Transform followed by Standard Scaling
rfm_features = ['recency_days', 'frequency_orders', 'monetary_value_usd']
df_log = np.log1p(df[rfm_features])

scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(df_log)

# Determine Optimal k via Elbow Inertia and Silhouette Score
k_range = range(2, 9)
inertias = []
silhouettes = []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(rfm_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(rfm_scaled, km.labels_))

fig, ax1 = plt.subplots(figsize=(10, 5))
ax1.plot(k_range, inertias, 'bo-', label='Inertia (Elbow)')
ax1.set_xlabel('Number of Clusters (k)')
ax1.set_ylabel('Inertia', color='blue')

ax2 = ax1.twinx()
ax2.plot(k_range, silhouettes, 'rs--', label='Silhouette Score')
ax2.set_ylabel('Silhouette Score', color='red')
plt.title("Cluster Evaluation: Inertia vs. Silhouette Coefficient", fontweight='bold')
plt.show()

In [ ]:
# Final Champion Model: K-Means (k=4) vs Agglomerative Hierarchical
km_final = KMeans(n_clusters=4, random_state=42, n_init=15)
df['cluster_kmeans'] = km_final.fit_predict(rfm_scaled)

agg_final = AgglomerativeClustering(n_clusters=4)
df['cluster_agg'] = agg_final.fit_predict(rfm_scaled)

print("=== Clustering Benchmark Matrix ===")
benchmarks = pd.DataFrame([
    {
        "Algorithm": "K-Means (k=4)",
        "Silhouette Score": round(silhouette_score(rfm_scaled, df['cluster_kmeans']), 4),
        "Calinski-Harabasz": round(calinski_harabasz_score(rfm_scaled, df['cluster_kmeans']), 1),
        "Davies-Bouldin": round(davies_bouldin_score(rfm_scaled, df['cluster_kmeans']), 4)
    },
    {
        "Algorithm": "Agglomerative (k=4)",
        "Silhouette Score": round(silhouette_score(rfm_scaled, df['cluster_agg']), 4),
        "Calinski-Harabasz": round(calinski_harabasz_score(rfm_scaled, df['cluster_agg']), 1),
        "Davies-Bouldin": round(davies_bouldin_score(rfm_scaled, df['cluster_agg']), 4)
    }
])
display(benchmarks)

In [ ]:
# Cluster Persona Profiling & Business Takeaways
cluster_summary = df.groupby('cluster_kmeans').agg(
    Customer_Count=('customer_id', 'count'),
    Avg_Recency_Days=('recency_days', 'mean'),
    Avg_Frequency_Orders=('frequency_orders', 'mean'),
    Avg_Monetary_USD=('monetary_value_usd', 'mean'),
    Total_Revenue=('monetary_value_usd', 'sum')
).reset_index()

cluster_summary['Revenue_Share_%'] = (cluster_summary['Total_Revenue'] / cluster_summary['Total_Revenue'].sum()) * 100
display(cluster_summary.round(2))

# Persona Tagging
persona_map = {
    cluster_summary.sort_values('Avg_Monetary_USD', ascending=False).iloc[0]['cluster_kmeans']: "VIP Champions",
    cluster_summary.sort_values('Avg_Monetary_USD', ascending=False).iloc[1]['cluster_kmeans']: "Loyal Steady",
    cluster_summary.sort_values('Avg_Recency_Days', ascending=False).iloc[0]['cluster_kmeans']: "Dormant / Churn",
}
remaining = [c for c in [0, 1, 2, 3] if c not in persona_map]
persona_map[remaining[0]] = "Recent Explorers"

df['Persona'] = df['cluster_kmeans'].map(persona_map)

plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=df, x='recency_days', y='monetary_value_usd',
    hue='Persona', palette='Set1', alpha=0.7, s=60
)
plt.yscale('log')
plt.title("Customer Persona Landscape: Recency vs Monetary Value", fontweight='bold')
plt.xlabel("Recency (Days since last purchase)")
plt.ylabel("Monetary Spend (USD, Log Scale)")
plt.show()